In [ ]:
from subprocess import run

sets = ["atlas", "chezod", "disprot", "pdbflex", "plddt", "softdis", "trizod"]
for train_set in sets:
    for test_set in sets:
        command = f"uv run ../predict.py ../data/{test_set}/test.fasta ../exported -t {train_set} -o results/{train_set}_{test_set}"
        print(command)
        run(command.split(" "))


# Setup

In [ ]:
import pandas as pd
from collections import defaultdict
import numpy as np
from torchmetrics.functional import spearman_corrcoef, auroc, average_precision
import torch
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import os
import matplotlib.colors as mcolors
import py3Dmol
import math
import polars as pl
from pathlib import Path
import json
import matplotlib

OUTPUT_DIR = Path("figures")
DPI = 600


results_path = "results"
datasets = ["trizod", "chezod", "softdis", "pdbflex", "atlas", "plddt", "disprot"]


# Load Data

In [ ]:
def read_results(input_dir: str) -> dict[str, list[float]]:
    results = {}
    for filename in sorted(os.listdir(input_dir)):
        if not filename.endswith(".caid"):
            continue
        filepath = os.path.join(input_dir, filename)
        with open(filepath) as f:
            lines = f.readlines()

        protein_id = lines[0].strip().lstrip(">")
        scores = []
        for line in lines[1:]:
            line = line.strip()
            if not line:
                continue
            parts = line.split("\t")
            scores.append(float(parts[2]))

        results[protein_id] = scores
    return results

def read_labels(test_file: str) -> dict[str, list[float]]:
    df = pd.read_json(test_file, lines=True)
    labels = {str(x["id"]): x["y"] for _, x in df.iterrows()}
    return labels

results = defaultdict(dict)

for test_ds in datasets:
    ref = read_labels(f"../data/{test_ds}/test.jsonl")
    for train_ds in datasets:
        res = read_results(f"{results_path}/{train_ds}_{test_ds}")
        results[train_ds][test_ds] = pd.DataFrame({'label': pd.Series(ref), 'pred': pd.Series(res)})
        

# Performance Measure Calculation

In [ ]:
def bootstrapped_std(preds, labels, function, n=100, **kwargs):
    metrics = []
    for _ in range(n):
        indices = torch.randint(0, len(labels), (len(labels),))
        sampled_preds = preds[indices]
        sampled_labels = labels[indices]

        metric = function(sampled_preds, sampled_labels, **kwargs)
        metrics.append(metric)
    
    return torch.tensor(metrics).std().item()

## Spearman

In [ ]:
neg = ["plddt", "chezod"]

In [ ]:
spearmans = defaultdict(dict)
spearman_stds = defaultdict(dict)

for train_ds in tqdm(datasets):
    for test_ds in tqdm(datasets):
        res = results[train_ds][test_ds]

        label = res["label"].tolist()
        label = [x for sublist in label for x in sublist]
        label = torch.tensor(label)
        mask = label != 999
        label = label[mask]

        pred = res["pred"].tolist()
        pred = [x for sublist in pred for x in sublist]
        pred = torch.tensor(pred)
        pred = pred[mask]

        if train_ds in neg:
            pred = -pred
        
        if test_ds in neg:
            label = -label

        spearmans[train_ds][test_ds] = spearman_corrcoef(pred, label)
        spearman_stds[train_ds][test_ds] = bootstrapped_std(pred, label, spearman_corrcoef)


## AUROC

In [ ]:
aurocs = {}
auroc_stds = {}

for train_ds in tqdm(datasets):
    res = results[train_ds]["disprot"]

    label = res["label"].tolist()
    label = [x for sublist in label for x in sublist]
    label = torch.tensor(label)
    mask = label != 999
    label = label[mask]

    pred = res["pred"].tolist()
    pred = [x for sublist in pred for x in sublist]
    pred = torch.tensor(pred)
    pred = pred[mask]

    if train_ds in neg:
        pred = -pred
    
    if test_ds in neg:
        label = -label


    aurocs[train_ds] = auroc(pred, label.to(int), task="binary")
    auroc_stds[train_ds] = bootstrapped_std(pred, label.to(int), auroc, task="binary")


## Average Precision

In [ ]:
ap = {}
ap_stds = {}

for train_ds in tqdm(datasets):
    res = results[train_ds]["disprot"]

    label = res["label"].tolist()
    label = [x for sublist in label for x in sublist]
    label = torch.tensor(label)
    mask = label != 999
    label = label[mask]

    pred = res["pred"].tolist()
    pred = [x for sublist in pred for x in sublist]
    pred = torch.tensor(pred)
    pred = pred[mask]

    if train_ds in neg:
        pred = -pred
    
    if test_ds in neg:
        label = -label


    ap[train_ds] = average_precision(pred, label.to(int), task="binary")
    ap_stds[train_ds] = bootstrapped_std(pred, label.to(int), auroc, task="binary")


## Merge

In [ ]:
res = defaultdict(dict)
res_stds = defaultdict(dict)

for train_ds in datasets:
    for test_ds in datasets:
        if test_ds == "disprot":
            res[train_ds]["disprot\n(AUROC)"] = aurocs[train_ds]
            res_stds[train_ds]["disprot\n(AUROC)"] = auroc_stds[train_ds]

            res[train_ds]["disprot\n(AP)"] = ap[train_ds]
            res_stds[train_ds]["disprot\n(AP)"] = ap_stds[train_ds]
        else:
            res[train_ds][test_ds] = spearmans[train_ds][test_ds]
            res_stds[train_ds][test_ds] = spearman_stds[train_ds][test_ds]


# Figure 2

In [ ]:
test_labels = ["trizod", "chezod", "softdis", "pdbflex", "atlas", "plddt", "disprot\n(AP)", "disprot\n(AUROC)"]

label_map = {"trizod": "TriZOD", "chezod": "CheZOD", "softdis": "SoftDis", "pdbflex": "PDBflex", "atlas": "ATLAS", "plddt": "pLDDT", "disprot\n(AP)": "DisProt\n(AP)", "disprot\n(AUROC)": "DisProt\n(AUROC)", "disprot": "DisProt"}


df = pd.DataFrame(res, index=test_labels, columns=datasets).map(lambda x: x.item()).T
df_stderr = pd.DataFrame(res_stds, index=test_labels, columns=datasets).T

df_left = df.iloc[:, :6]
df_right = df.iloc[:, 6:]

df_left.columns = df_left.columns.map(label_map)
df_left.index = df_left.index.map(label_map)
df_right.columns = df_right.columns.map(label_map)
df_right.index = df_right.index.map(label_map)

df_left["Average"] = df_left.mean(axis=1)
df_stderr["Average"] = df_stderr.mean(axis=1)


def get_annots(dataframe, stderr_df):
    return np.array([[f"{dataframe.iloc[i, j]:.3f}\n({stderr_df.iloc[i, j]:.3f})" 
                      for j in range(dataframe.shape[1])] 
                     for i in range(dataframe.shape[0])])

annot_left = get_annots(df_left, df_stderr.iloc[:, :7])
annot_right = get_annots(df_right, df_stderr.iloc[:, 7:])

FONT_SIZE_TITLE = 45
FONT_SIZE_AXIS = 35
FONT_SIZE_TICKS = 32
FONT_SIZE_ANNOT =30  
FONT_SIZE_CBAR = 30  

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(30, 20), 
                               sharey=True, 
                               gridspec_kw={'width_ratios': [6, 2]})

plt.subplots_adjust(wspace=0.04, right=0.85, left=0.12, bottom=0.12) 

cbar_ax1 = fig.add_axes([0.87, 0.55, 0.02, 0.3]) 
cbar_ax2 = fig.add_axes([0.87, 0.18, 0.02, 0.15])

sns.heatmap(df_left, annot=annot_left, fmt="", cmap="YlGnBu", ax=ax1,
            annot_kws={"size": FONT_SIZE_ANNOT, "weight": "bold"},
            cbar_ax=cbar_ax1,
            cbar_kws={'label': 'Spearman Correlation'})

sns.heatmap(df_right, annot=annot_right, fmt="", cmap="YlOrRd", ax=ax2,
            annot_kws={"size": FONT_SIZE_ANNOT, "weight": "bold"},
            cbar_ax=cbar_ax2,
            cbar_kws={'label': 'AUROC / AP'})

ax1.set_ylabel("Train Set", fontsize=FONT_SIZE_AXIS, labelpad=20)
ax1.set_xlabel("") 
ax2.set_xlabel("") 

fig.text(0.48, 0.05, "Test Set", ha='center', fontsize=FONT_SIZE_AXIS)

ax1.tick_params(axis='both', labelsize=FONT_SIZE_TICKS)
ax2.tick_params(axis='both', labelsize=FONT_SIZE_TICKS)

for c_ax in [cbar_ax1, cbar_ax2]:
    c_ax.yaxis.label.set_size(FONT_SIZE_CBAR-3)
    c_ax.tick_params(labelsize=FONT_SIZE_CBAR - 4)

plt.savefig(OUTPUT_DIR / "heatmap.png", dpi=DPI, bbox_inches="tight")
plt.show()


# Figure 3

## Structures

In [ ]:
def matrix_to_xyz(m):
    y_rad = math.asin(max(-1.0, min(1.0, m[2])))

    x_rad = math.atan2(-m[6], m[10])
    z_rad = math.atan2(-m[1], m[0])

    return (math.degrees(x_rad),
            math.degrees(y_rad),
            math.degrees(z_rad))


def plot_structure(pdb, values, x, y, z):

    with open(pdb, 'r') as f:
        pdb_data = f.read()

    cmap = plt.get_cmap('cool')
    norm = mcolors.Normalize(vmin=min(x for x in values if x is not None),
                            vmax=max(x for x in values if x is not None))

    def get_color(val):
        if val is None:
            return "#D3D3D3"
        return mcolors.to_hex(cmap(norm(val)))

    view = py3Dmol.view(width=800, height=600)
    view.addModel(pdb_data, 'pdb')

    view.setStyle({}, {})
    view.addStyle({'chain': 'A'}, {'cartoon': {'color': '#D3D3D3'}})

    for i, val in enumerate(values):
        res_index = i + 1
        hex_color = get_color(val)
        view.addStyle({'chain': 'A', 'resi': res_index},
                    {'cartoon': {'color': hex_color}})

    view.rotate(x, 'x')
    view.rotate(y, 'y')
    view.rotate(z, 'z')

    view.zoomTo()
    view.show()
    view.png()


m = [-0.64709,0.75887,0.07345,12.294,-0.38412,-0.24128,-0.8912,-134.68,-0.65858,-0.6049,0.44762,72.124]
x, z, y = matrix_to_xyz(m)


In [ ]:
trizod_values = [None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,0.0105,0.027,0.0266,0.0368,0.0368,0.0699,0.0612,0.055,0.042,0.056,0.0362,0.016,0.0099,0.0429,0.0625,0.1126,0.0905,0.0929,0.0842,0.1098,0.1171,0.0689,0.0424,0.0131,0.0186,0.0561,0.0561,0.0533,0.0083,0.0326,0.0388,0.0567,0.0876,0.096,0.0975,0.0758,0.0804,0.0585,0.0429,0.0232,0.0453,0.0573,0.1695,0.1706,0.1601,0.053,0.0256,0.0025,0.0138,0.0081,0.0054,0.0331,0.0474,0.0872,0.0771,0.1167,0.1408,0.1182,0.0723,0.0646,0.0758,0.0845,0.0643,0.0745,0.1271,0.0876,0.064,0.0261,0.0497,0.0653,0.0384,0.061,0.0674,0.0674,0.076,0.0618,0.2342,0.0,0.0346,0.051,0.0971,0.1512,0.1037,0.0912,0.0347,0.1142,0.0826,0.1286,0.099,0.0802,0.0584,0.0146,0.0175,0.035,0.1147,0.227,0.3447,0.1517,0.1019,0.0506,0.0824,0.0647,0.048,0.0375,0.0772,0.0651,0.0667,0.0326,0.0747,0.1249,0.115,0.1242,0.0393,0.0765,0.0451,0.0531,0.0373,0.039,0.036,0.0232,0.0,0.0,0.2313,0.2467,0.2793,0.254,0.3174,0.2153,0.2474,0.1136,0.0837,0.0596,0.0827,0.066,0.0435,0.0178,0.044,0.0897,0.0728,0.0493,0.0159,0.0168,0.0469,0.0718,0.1152,0.0743,0.0942,0.0831,0.0911,0.087,0.1128,0.1358,0.1369,0.1909,0.2131,0.1986,0.1257,0.0905,0.0308,0.0505,0.0316,0.0492,0.0366,0.0682,0.1036,0.1213,0.1213,0.0891,0.0651,0.0673,0.0788,0.1012,0.0704,0.0401,0.0146,0.0338,0.0351,0.0465,0.0074,0.0387,0.0465,0.1184,0.7845,0.756,0.7559,0.7452,None]
trizod_values = [min(x, 0.4) if x is not None else None for x in trizod_values]
plot_structure("merb.pdb", trizod_values, x, y, z)

In [ ]:
disprot_values = [1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0]
plot_structure("merb.pdb", disprot_values, x, y, z)

In [ ]:
DATA_PATH = Path("../data")
NULL_VALUE = 999
PROTEIN_ID_COLUMN = "protein_id"
POSITION_COLUMN = "position"
SEQUENCE_COLUMN_LABEL = "residue"
SCORE_COLUMN_LABEL = "score"

def load_jsonl(dataset_name: str, split_name: str) -> list[dict[str, str | list[str, float | int]]]:
    return pl.from_records([
        {
            PROTEIN_ID_COLUMN: str(x["id"]),
            POSITION_COLUMN: i,
            SEQUENCE_COLUMN_LABEL: residue,
            SCORE_COLUMN_LABEL: float(score)
        }
        for x in [json.loads(line) for line in (DATA_PATH / f"{dataset_name}/{split_name}.jsonl").open()]
        for i, (residue, score) in enumerate(zip(x["x_0"], x["y"]), start=1)
    ]).with_columns(
            pl.col(SCORE_COLUMN_LABEL).replace_strict(NULL_VALUE, None, default=pl.col(SCORE_COLUMN_LABEL))
        )


def get_dataset_size(dataset_df: pl.DataFrame) -> dict[str, int]:
    df = dataset_df.filter(~pl.col("score").is_nan())
    return {
        "proteins": df.select(pl.col("protein_id")).unique().height,
        "residues": df.height
    }

def format_k(value: int) -> str:
    if value == 0:
        return "0"

    suffixes = ["", "K", "M", "B", "T"]
    idx = 0
    scaled = float(value)

    # Scale down
    while abs(scaled) >= 1000 and idx < len(suffixes) - 1:
        scaled /= 1000.0
        idx += 1

    # Logic:
    # 1. If >= 100, force integer (e.g. 122.1 -> 122)
    # 2. If < 100, format to 1 decimal, then strip '.0' if present
    if abs(scaled) >= 100:
        formatted_num = f"{scaled:.0f}"
    else:
        formatted_num = f"{scaled:.1f}"
        if formatted_num.endswith(".0"):
            formatted_num = formatted_num[:-2]

    return f"{formatted_num}{suffixes[idx]}"

In [ ]:
train_df = load_jsonl("trizod", "train")
valid_df = load_jsonl("trizod", "valid")
test_df = load_jsonl("trizod", "test")
trizod_df = pl.concat([train_df, valid_df, test_df])

train_df = load_jsonl("chezod", "train")
valid_df = load_jsonl("chezod", "valid")
test_df = load_jsonl("chezod", "test")
chezod_df = pl.concat([train_df, valid_df, test_df])


train_df = load_jsonl("atlas", "train")
valid_df = load_jsonl("atlas", "valid")
test_df = load_jsonl("atlas", "test")
atlas_df = pl.concat([train_df, valid_df, test_df])

disprot_df = load_jsonl("disprot", "DP00575")

train_df = load_jsonl("pdbflex", "train")
valid_df = load_jsonl("pdbflex", "valid")
test_df = load_jsonl("pdbflex", "test")
pdbflex_df = pl.concat([train_df, valid_df, test_df])

train_df = load_jsonl("plddt", "train")
valid_df = load_jsonl("plddt", "valid")
test_df = load_jsonl("plddt", "test")
plddt_df = pl.concat([train_df, valid_df, test_df])

train_df = load_jsonl("softdis", "train")
valid_df = load_jsonl("softdis", "valid")
test_df = load_jsonl("softdis", "test")
softdis_df = pl.concat([train_df, valid_df, test_df])

In [ ]:
plt.style.use("ggplot")
sns.set_context("paper", font_scale=1.2)
# Override grey text color of ggplot theme
matplotlib.rcParams.update({
    "text.color": "black",
    "axes.labelcolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
    "axes.titlecolor": "black"
})
# Get colors for custom stuff
COLORS = plt.rcParams["axes.prop_cycle"].by_key()['color']    
sns.color_palette(COLORS)

In [ ]:
# 95% sequence identity
pdbflex_id = "3fn8A"
atlas_id = "3f0o_B"
atlas_id_2 = "3f0p_A"
chezod_id = "6047"
disprot_id = "DP00575"
plddt_id = "DP00575"
softdis_id = "3f2g_A"

In [ ]:
# Remove hist tag
hist_tag = "HHHHHH"

# 214 residues, C-terminal His6-tag removed
protein_pdbflex = pdbflex_df.filter(pl.col("protein_id") == pdbflex_id).filter(pl.col("position") < pl.col("position").max() - len(hist_tag) + 1)
# 212 residues
protein_atlas = atlas_df.filter(pl.col("protein_id") == atlas_id)
# 211 residues, first 25 are NaN
protein_chezod = chezod_df.filter(pl.col("protein_id") == chezod_id)
# 212 residues
protein_disprot = disprot_df.filter(pl.col("protein_id") == disprot_id)
# 212 residues
protein_plddt = plddt_df.filter(pl.col("protein_id") == plddt_id).with_columns((1 - (pl.col("score") / 100)).alias("score"))
# 214 residues, C-terminal His6-tag removed, C160S
protein_softdis = softdis_df.filter(pl.col("protein_id") == softdis_id).filter(pl.col("position") < pl.col("position").max() - len(hist_tag) + 1)

raw = {"ID":"6047_1_1_1","entryID":"6047","stID":"1","entity_assemID":"1","entityID":"1","entity_name":"Organomercurial Lyase","exp_method":"NMR","exp_method_subtype":None,"citation_DOI":"","citation_title":"1H, 15N, and 13C resonance assignment of the 23 kDa organomercurial lyase MerB \nin its free and mercury-bound forms\n","ionic_strength":0.1,"pH":7.5,"temperature":300.0,"off_C":0.0,"off_CA":0.0,"off_CB":0.0,"off_H":0.0,"off_HA":0.0,"off_HB":0.0,"off_N":0.0,"bbshift_positions_post":179.0,"bbshift_types_post":7.0,"total_bbshifts":1128.0,"seq":"MKLAPYILELLTSVNRTNGTADLLVPLLRELAKGRPVSRTTLAGILDWPAERVAAVLEQATSTEYDKDGNIIGYGLTLRETSYVFEIDDRRLYAWCALDTLIFPALIGRTARVSSHCAATGAPVSLTVSPSEIQAVEPAGMAVSLVLPQEAADVRQSFCCHVHFFASVPTAEDWASKHQGLEGLAIVSVHEAFGLGQEFNRHLLQTMSSRTP","k":[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,5,11,17,19,19,19,19,19,18,17,16,18,19,21,21,21,21,21,19,19,19,21,20,18,18,19,20,20,20,21,21,21,21,21,21,21,21,21,21,21,20,18,18,17,19,19,21,19,12,7,8,15,19,20,19,20,20,21,21,21,21,21,21,21,21,20,20,20,21,20,14,14,13,15,15,14,11,4,3,10,16,20,18,19,19,21,21,21,21,21,21,21,21,21,21,19,18,16,18,19,21,21,21,20,18,18,19,21,21,21,21,20,18,18,17,19,19,21,21,19,14,7,2,5,12,19,21,20,20,16,16,16,20,19,19,19,21,21,21,21,21,21,20,18,18,19,21,21,21,21,21,21,21,21,19,19,19,17,15,15,19,21,21,21,21,21,21,21,19,19,17,18,15,13,14,17,18,12,10,13,18,13,6,5,12,18,13,6],"zscores":[None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,11.4962,13.8176,14.6123,14.3421,14.3415,13.5314,13.7376,13.5191,13.4474,12.7322,13.9797,14.9029,15.8425,14.9061,14.4007,13.2328,13.7298,13.0146,13.2053,12.6533,13.1355,13.9026,13.8233,14.5884,14.8308,14.2154,14.2154,14.2851,15.8923,15.1866,15.0181,14.5475,13.7968,13.6048,13.5699,14.0765,13.9666,14.5013,14.9071,15.0815,13.7523,13.4648,10.8677,11.4592,11.6558,14.6428,14.6389,12.1853,9.1557,9.8783,13.5275,14.4386,14.4355,13.1389,13.7105,12.8315,12.6361,13.1112,14.1595,14.3502,14.0751,13.8684,14.3561,14.1072,12.6133,13.4679,14.0216,15.3697,14.3758,11.7307,12.297,11.3938,12.0927,12.0927,11.517,10.4813,4.8885,6.2632,10.492,12.8448,13.2556,11.5132,12.7819,13.0517,15.1296,13.1996,13.9138,12.8906,13.5363,13.9701,14.5059,15.7027,15.6176,15.121,12.5505,10.1863,7.8873,11.5049,12.821,14.7044,13.9176,14.3472,14.4187,13.9467,13.0107,13.644,14.2978,15.1857,14.1022,12.9689,12.8666,12.0312,13.9007,12.6642,14.1308,13.9322,15.0574,15.0126,14.3643,12.6417,9.3998,5.1996,5.4488,8.089,9.5988,10.5159,9.228,10.9343,9.2994,11.5524,12.1424,14.1298,13.2399,13.623,14.1715,15.6082,14.8782,13.7474,14.1487,14.7385,15.6628,15.2644,13.714,13.1318,12.5401,14.1114,13.6442,13.9027,13.7156,13.8101,13.2291,12.7391,12.7161,11.0912,10.6992,10.9539,11.6681,11.6278,12.8991,13.9973,15.2145,14.7415,15.0758,14.2615,13.4321,13.0453,13.045,13.0983,13.6443,12.8639,12.9753,11.4221,11.2114,12.2598,14.1415,14.0406,11.4654,10.2731,12.5566,13.9151,11.6869,7.1041,1.0668,1.8817,2.2535,2.1062,None],"gscores":[None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,0.0105,0.027,0.0266,0.0368,0.0368,0.0699,0.0612,0.055,0.042,0.056,0.0362,0.016,0.0099,0.0429,0.0625,0.1126,0.0905,0.0929,0.0842,0.1098,0.1171,0.0689,0.0424,0.0131,0.0186,0.0561,0.0561,0.0533,0.0083,0.0326,0.0388,0.0567,0.0876,0.096,0.0975,0.0758,0.0804,0.0585,0.0429,0.0232,0.0453,0.0573,0.1695,0.1706,0.1601,0.053,0.0256,0.0025,0.0138,0.0081,0.0054,0.0331,0.0474,0.0872,0.0771,0.1167,0.1408,0.1182,0.0723,0.0646,0.0758,0.0845,0.0643,0.0745,0.1271,0.0876,0.064,0.0261,0.0497,0.0653,0.0384,0.061,0.0674,0.0674,0.076,0.0618,0.2342,0.0,0.0346,0.051,0.0971,0.1512,0.1037,0.0912,0.0347,0.1142,0.0826,0.1286,0.099,0.0802,0.0584,0.0146,0.0175,0.035,0.1147,0.227,0.3447,0.1517,0.1019,0.0506,0.0824,0.0647,0.048,0.0375,0.0772,0.0651,0.0667,0.0326,0.0747,0.1249,0.115,0.1242,0.0393,0.0765,0.0451,0.0531,0.0373,0.039,0.036,0.0232,0.0,0.0,0.2313,0.2467,0.2793,0.254,0.3174,0.2153,0.2474,0.1136,0.0837,0.0596,0.0827,0.066,0.0435,0.0178,0.044,0.0897,0.0728,0.0493,0.0159,0.0168,0.0469,0.0718,0.1152,0.0743,0.0942,0.0831,0.0911,0.087,0.1128,0.1358,0.1369,0.1909,0.2131,0.1986,0.1257,0.0905,0.0308,0.0505,0.0316,0.0492,0.0366,0.0682,0.1036,0.1213,0.1213,0.0891,0.0651,0.0673,0.0788,0.1012,0.0704,0.0401,0.0146,0.0338,0.0351,0.0465,0.0074,0.0387,0.0465,0.1184,0.7845,0.756,0.7559,0.7452,None]}
# 211 residues, first 25 are NaN
protein_trizod = pl.from_records([{"protein_id": raw["entryID"], "position": i, "residue": residue, "score": score} for i, (residue, score) in enumerate(zip(raw["seq"], raw["gscores"]), start=1)])

In [ ]:
c1 = COLORS[0]
c2 = COLORS[-1]

def create_colormaps(color1, color2):
    # Create the gradient colormap
    # LinearSegmentedColormap creates a smooth transition
    cmap_gradient = matplotlib.colors.LinearSegmentedColormap.from_list("custom_gradient", [color1, color2])

    # Create the qualitative version with 5 steps
    # We sample 5 equidistant colors from the gradient
    colors_qualitative = cmap_gradient(np.linspace(0, 1, 5))
    cmap_qualitative = matplotlib.colors.ListedColormap(colors_qualitative)

    return cmap_gradient, cmap_qualitative

# Generate the colormaps
cmap_gradient, cmap_qualitative = create_colormaps(c1, c2)

In [ ]:
mosaic_layout = [
    *[[f"A{i}", "B"] for i in range(3)],
    *[[f"A{i + 3}", "C"] for i in range(3)],
    ["A6", "."],
]

fig, ax_dict = plt.subplot_mosaic(
    mosaic_layout, 
    figsize=(12, 6), 
    constrained_layout=True,
    width_ratios=[4, 1],
    gridspec_kw={"wspace": 0.01, "hspace": .02}
)

lineplot_axes = [ax_dict[f"A{i}"] for i in range(7)]

# 4. Handle "Share X" manually (subplot_mosaic doesn"t have a sharex arg)
# We link all lineplots to the bottom-most lineplot
for ax in lineplot_axes[:-1]:
    ax.sharex(lineplot_axes[-1])
    plt.setp(ax.get_xticklabels(), visible=False)

trizod_structure_image = "structure_trizod.png"
image = matplotlib.image.imread(trizod_structure_image)
h, w = image.shape[:2]
ax_dict["B"].imshow(image, aspect="equal")
ax_dict["B"].axis("off")

disprot_structure_image = "structure_disprot.png"
image = matplotlib.image.imread(disprot_structure_image)
h, w = image.shape[:2]
ax_dict["C"].imshow(image, aspect="equal")
ax_dict["C"].axis("off")

PRIMARY_LINE_COLOR=COLORS[0]
REFERENCE_LINE_KWARGS = {"linestyle": "--", "linewidth": 1}
FILL_KWARGS = {"alpha": .5}

ax = lineplot_axes[0]
sns.lineplot(data=protein_trizod, x="position", y="score", ax=ax, c=PRIMARY_LINE_COLOR)
ax.set_ylabel("TriZOD\nG-Score", rotation=0, verticalalignment="center", horizontalalignment="right")
ax.set_ylim([0, 1])
ax.set_yticks([0, 1])
ax.fill_between(protein_trizod["position"], protein_trizod["score"], **FILL_KWARGS)
#ax.axhline(.4, **REFERENCE_LINE_KWARGS, c=COLORS[0])

ax = lineplot_axes[1]
sns.lineplot(data=protein_chezod, x="position", y="score", ax=ax, c=PRIMARY_LINE_COLOR)
ax.set_ylabel("CheZOD\nZ-Score", rotation=0, verticalalignment="center", horizontalalignment="right")
ax.set_ylim([0, 16])
ax.set_yticks([0, 16])
ax.invert_yaxis()
# TODO fix inversion of fill
ax.fill_between(protein_chezod["position"], protein_chezod["score"], 16, **FILL_KWARGS)
#ax.axhline(3, **REFERENCE_LINE_KWARGS, c=COLORS[0])
#ax.axhline(8, **REFERENCE_LINE_KWARGS, c=COLORS[1])

ax = lineplot_axes[2]
sns.lineplot(data=protein_softdis, x="position", y="score", ax=ax, c=PRIMARY_LINE_COLOR)
ax.set_ylabel("SoftDis", rotation=0, verticalalignment="center", horizontalalignment="right")
ax.set_ylim([0, 1])
ax.set_yticks([0, 1])
ax.fill_between(protein_softdis["position"], protein_softdis["score"], **FILL_KWARGS)

ax = lineplot_axes[3]
sns.lineplot(data=protein_plddt, x="position", y="score", ax=ax, c=PRIMARY_LINE_COLOR)
ax.set_ylabel("1 - pLDDT/100", rotation=0, verticalalignment="center", horizontalalignment="right")
ax.set_ylim([0, 1])
ax.set_yticks([0, 1])
ax.fill_between(protein_plddt["position"], protein_plddt["score"], **FILL_KWARGS)

ax = lineplot_axes[4]
sns.lineplot(data=protein_pdbflex, x="position", y="score", ax=ax, c=PRIMARY_LINE_COLOR)
ax.set_ylabel("PDBflex\nRMSD", rotation=0, verticalalignment="center", horizontalalignment="right")
ax.fill_between(protein_pdbflex["position"], protein_pdbflex["score"], **FILL_KWARGS)
ax.set_yticks([0, 3.5])

ax = lineplot_axes[5]
sns.lineplot(data=protein_atlas, x="position", y="score", ax=ax, c=PRIMARY_LINE_COLOR)
ax.set_ylabel("ATLAS\nRMSF", rotation=0, verticalalignment="center", horizontalalignment="right")
ax.fill_between(protein_atlas["position"], protein_atlas["score"], **FILL_KWARGS)
ax.set_yticks([0, 2.2])

ax = lineplot_axes[6]
sns.lineplot(data=protein_disprot, x="position", y="score", ax=ax, c=PRIMARY_LINE_COLOR)
ax.set_ylabel("DisProt", rotation=0, verticalalignment="center", horizontalalignment="right")
ax.fill_between(protein_disprot["position"], protein_disprot["score"], **FILL_KWARGS)
ax.set_ylim([0, 1])
ax.set_yticks([0, 1])

plt.xlabel("Residue Number")
plt.xlim([0, 212])

for i, ax in enumerate(lineplot_axes):
    ax.tick_params(axis="y", length=0, pad=4)
    if i != 1:
        bottom_label, top_label = ax.get_yticklabels()
    else:
        top_label, bottom_label = ax.get_yticklabels()
        
    top_label.set_verticalalignment("top")
    bottom_label.set_verticalalignment("bottom")
        
    
    if i != 6:
        ax.tick_params(axis="x", labelbottom=False, length=0)


lineplot_axes[0].text(
    -0.08, 1.1, "A", 
    transform=lineplot_axes[0].transAxes, 
    size=16, weight="bold", va='bottom', ha='right'
)

# Label B: Top image
ax_dict["B"].text(
    -0.1, 1.0, "B", 
    transform=ax_dict["B"].transAxes, 
    size=16, weight="bold", va='bottom', ha='right'
)

# Label C: Bottom image
ax_dict["C"].text(
    -0.1, 1.0, "C", 
    transform=ax_dict["C"].transAxes, 
    size=16, weight="bold", va='bottom', ha='right'
)

plt.savefig(OUTPUT_DIR / "protein_case_study.png", dpi=DPI, bbox_inches="tight")
plt.show()